V10 Мат модель для нахождения коэффициента влияния гендера на ЗП

In [21]:
import pandas as pd

modelv1 = pd.read_excel("../../data/processed/v1_clean_base.xlsx")

model v10 (линейная регрессия по параметрам: salary, gender, experience, education_type, region_code, industry_code)

In [22]:
import pandas as pd
import statsmodels.formula.api as smf

cols = ["salary", "gender", "experience", "education_type", "region_code", "industry_code"]
data = modelv1[cols].dropna().copy()

min_n = 20
results = []

for (region, industry), g in data.groupby(["region_code", "industry_code"]):
    if len(g) < min_n:
        continue
    
    if g["gender"].nunique() < 2:
        continue

    model = smf.ols(
        "salary ~ C(gender) + experience + C(education_type)",
        data=g
    ).fit()

    for name, coef in model.params.items():
        if "C(gender)" in name:
            results.append({
                "region_code": region,
                "industry_code": industry,
                "n": len(g),
                "term": name,
                "coef": coef,
                "pvalue": model.pvalues[name],
                "r2": model.rsquared
            })

v10_result = pd.DataFrame(results)

# Проверка данных

print(v10_result.head())

   region_code industry_code   n                  term         coef    pvalue  \
0            2      DeskWork  83  C(gender)[T.Мужской]  7648.037901  0.016283   
1            2     Education  34  C(gender)[T.Мужской] -4788.373904  0.066882   
2            2      Industry  21  C(gender)[T.Мужской]  8314.694403  0.103243   
3            2      Medicine  20  C(gender)[T.Мужской]     9.505784  0.998330   
4            2         Sales  40  C(gender)[T.Мужской]  1522.351963  0.723514   

         r2  
0  0.178857  
1  0.488915  
2  0.171443  
3  0.502595  
4  0.471221  


Сохранение результатов model v10

In [23]:
v10_result.to_excel("../../data/processed/v10_result.xlsx", index=False)